# Music Annotation Project V8.8
## Notebook 52F: Full-Development SAFE_REMOVE Fit and Freeze

**Runtime:** CPU. In Google Colab select **Runtime > Run all**.

This notebook completes the missing deployable-scoring step between Notebook 50 and the V8.8 operational builder.

### Fixed contract
- Resolve the authoritative Notebook 50 run through a passed Notebook 52 policy record.
- Require Notebook 50's REMOVE-only verdict and selected REMOVE family `local_nonlinear`.
- Recreate the exact Notebook 50 constructor without adding, removing or changing a modelling parameter.
- Fit one estimator on the complete permissible Notebook 50 development population.
- Fit one isotonic calibrator from the complete source-group OOF raw REMOVE scores and REMOVE truth labels.
- Preserve the frozen rule: threshold `0.995`, region mean `0.995`, 8-connectivity, minimum run `2`, and every asserted region pixel above threshold.
- Preserve M3 fallback and keep SAFE_PROTECT disabled.
- Produce a content-addressed estimator/calibrator bundle and deterministic replay fixture.

The estimator created here is a **post-development deployment refit**. It is not represented as a historical fold estimator or as reproducing Notebook 50 OOF probabilities.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy pyarrow joblib scikit-learn opencv-python-headless

## 1. Configuration and immutable constants

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, sys, platform, zipfile
import numpy as np
import pandas as pd
import cv2, joblib, sklearn
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N52_ROOT=ROOT/'m5_remove_policy_freeze_v8_8a'
OUTPUT_ROOT=ROOT/'m5_remove_full_development_freeze_v8_8f'
NOTEBOOK52_RUN_OVERRIDE=None
NOTEBOOK50_RUN_OVERRIDE=None

SEED=42
CONNECTIVITY=8
REMOVE_PROB_THRESHOLD=0.995
REMOVE_REGION_MIN_MEAN=0.995
REMOVE_REGION_MIN_PIXEL_FRACTION=1.00
MIN_REMOVE_RUN=2
EXPECTED_N50_VERDICT='M5_REMOVE_ONLY_READY_FOR_NOTEBOOK51_CROSSOVER'
EXPECTED_N52_VERDICT='PASS_M5_REMOVE_FREEZE_READY_FOR_OPERATIONAL_BUILDER'
EXPECTED_REMOVE_MODEL='local_nonlinear'
EXPECTED_RAW_REMOVE_COLUMN='raw_remove_local_nonlinear'

RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook52f_v8_8_full_development_remove_freeze'
OUT=OUTPUT_ROOT/RUN_ID; AUDIT=OUT/'audit'; BUNDLE=OUT/'full_development_remove'; REPLAY=BUNDLE/'replay'; TRAINING=BUNDLE/'training_record'; UPLOAD=OUT/'upload'
for p in [OUT,AUDIT,BUNDLE,REPLAY,TRAINING,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)

## 2. Resolve and verify Notebook 52 and Notebook 50

In [ ]:
def require(x,msg):
    if not bool(x): raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def object_hash(x): return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def read_json(p): return json.loads(Path(p).read_text())
def atomic_json(p,x):
    t=Path(str(p)+'.tmp'); t.write_text(json.dumps(x,indent=2,sort_keys=True,default=str)); t.replace(p)
def complete_runs(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p.resolve())
    return sorted({str(p):p for p in hits}.values(),key=lambda p:p.stat().st_mtime)

N52_REQUIRED=['notebook52_verdict.json','policy_bundle/policy_manifest.json','tables/m5_remove_golden_master_semantic_hashes.csv']
if NOTEBOOK52_RUN_OVERRIDE:
    N52=Path(NOTEBOOK52_RUN_OVERRIDE)
else:
    hits=complete_runs(N52_ROOT,N52_REQUIRED); require(hits,'No complete Notebook 52 run found.'); N52=hits[-1]
require(all((N52/r).exists() for r in N52_REQUIRED),f'Incomplete Notebook 52: {N52}')
verdict52=read_json(N52/'notebook52_verdict.json'); policy52=read_json(N52/'policy_bundle'/'policy_manifest.json')
require(verdict52.get('verdict')==EXPECTED_N52_VERDICT,f'Notebook 52 did not pass: {verdict52}')
require(policy52.get('policy')=='M5_REMOVE','Notebook 52 policy is not M5_REMOVE.')
require(policy52.get('safe_protect_runtime_enabled') is False,'SAFE_PROTECT must remain disabled.')
N50=Path(NOTEBOOK50_RUN_OVERRIDE) if NOTEBOOK50_RUN_OVERRIDE else Path(policy52['authoritative_notebook50_run'])
N50_REQUIRED=['experiment_identity.json','notebook50_verdict.json','selected_models.json','inputs/predictor_schema.json','tables/all_m3_oof_predictions_and_decisions.parquet','tables/asymmetric_target_summary.csv']
require(N50.exists(),f'Notebook 50 run missing: {N50}')
missing=[r for r in N50_REQUIRED if not (N50/r).exists()]; require(not missing,f'Notebook 50 incomplete: {missing}')
print('Notebook 52:',N52); print('Notebook 50:',N50); print('Source policy ID:',policy52['policy_id'])

## 3. Enforce selected family, schema and exact constructor

In [ ]:
identity50=read_json(N50/'experiment_identity.json'); verdict50=read_json(N50/'notebook50_verdict.json'); selected_models=read_json(N50/'selected_models.json'); schema=read_json(N50/'inputs'/'predictor_schema.json')
verdict_text=str(verdict50.get('verdict',verdict50.get('decision',verdict50.get('status',''))))
predictors=schema.get('ordered_predictors',schema.get('predictors',schema.get('ordered_features',[])))
checks={
 'notebook52_passed':verdict52.get('verdict')==EXPECTED_N52_VERDICT,
 'notebook50_remove_only_verdict':verdict_text==EXPECTED_N50_VERDICT,
 'remove_model_exact':selected_models.get('REMOVE')==EXPECTED_REMOVE_MODEL,
 'predictor_schema_nonempty':len(predictors)>0,
 'safe_protect_disabled':policy52.get('safe_protect_runtime_enabled') is False,
 'notebook52_no_retraining':policy52.get('retraining_performed') is False,
 'notebook52_no_recalibration':policy52.get('recalibration_performed') is False,
 'notebook52_no_threshold_change':policy52.get('threshold_changed') is False,
}
require(all(checks.values()),f'Source control failure: {checks}')

def make_full_development_remove_estimator():
    return Pipeline([
      ('impute',SimpleImputer(strategy='median')),
      ('model',HistGradientBoostingClassifier(max_iter=300,max_leaf_nodes=15,min_samples_leaf=20,l2_regularization=2.0,random_state=SEED))
    ])
constructor={'selected_model_label':'local_nonlinear','pipeline':[{'step':'impute','class':'SimpleImputer','parameters':{'strategy':'median'}},{'step':'model','class':'HistGradientBoostingClassifier','parameters':{'max_iter':300,'max_leaf_nodes':15,'min_samples_leaf':20,'l2_regularization':2.0,'random_state':42}}],'source':'Notebook 50 make_models() local_nonlinear arm'}
atomic_json(TRAINING/'exact_model_constructor.json',constructor)
print(json.dumps(checks,indent=2)); print('Predictors:',len(predictors))

## 4. Load exact Notebook 50 development population

In [ ]:
source_table=N50/'tables'/'all_m3_oof_predictions_and_decisions.parquet'; df=pd.read_parquet(source_table)
require({'page','source_group','y','x'}.issubset(df.columns),'Notebook 50 table lacks identity columns.')
require(not df.duplicated(['page','y','x']).any(),'Duplicate page/y/x rows.')
label_col=next((c for c in ['truth_class','diagnostic_class','class'] if c in df.columns),None); require(label_col,'No truth-side class column.')
df[label_col]=df[label_col].astype(str).str.upper()
require(set(df[label_col].unique()).issubset({'REMOVE','PROTECT','UNCERTAIN','RECONSTRUCT'}),f'Unexpected classes: {sorted(df[label_col].unique())}')
missing=[c for c in predictors if c not in df.columns]; require(not missing,f'Missing predictors: {missing}')
require(EXPECTED_RAW_REMOVE_COLUMN in df.columns,f'Missing {EXPECTED_RAW_REMOVE_COLUMN}')
X=df[predictors].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan); y=(df[label_col]=='REMOVE').astype(int); raw_oof=pd.to_numeric(df[EXPECTED_RAW_REMOVE_COLUMN],errors='coerce').to_numpy(float)
require(np.isfinite(raw_oof).all(),'Non-finite raw OOF REMOVE scores.'); require(y.nunique()==2,'REMOVE target needs two classes.')
population={'rows':len(df),'pages':df.page.astype(str).nunique(),'source_groups':df.source_group.astype(str).nunique(),'remove_positive_rows':int(y.sum()),'remove_negative_rows':int((1-y).sum()),'ordered_predictors':predictors,'label_column':label_col,'raw_oof_remove_column':EXPECTED_RAW_REMOVE_COLUMN,'source_table':str(source_table),'source_table_sha256':sha256_file(source_table)}
atomic_json(TRAINING/'development_population.json',population)
df[['page','source_group','y','x']].to_parquet(TRAINING/'development_roster.parquet',index=False)
shutil.copy2(N50/'inputs'/'predictor_schema.json',BUNDLE/'predictor_schema.json'); shutil.copy2(N50/'selected_models.json',TRAINING/'selected_models.json'); shutil.copy2(N50/'notebook50_verdict.json',TRAINING/'notebook50_verdict.json'); shutil.copy2(N50/'experiment_identity.json',TRAINING/'source_notebook50_identity.json')
print(json.dumps(population,indent=2))

## 5. Fit full-development estimator and OOF-evidence calibrator

In [ ]:
remove_estimator=make_full_development_remove_estimator(); remove_estimator.fit(X,y)
# Fit calibration to complete source-group OOF ranking scores, not in-sample scores from the new refit.
remove_calibrator=IsotonicRegression(out_of_bounds='clip').fit(raw_oof,y.to_numpy())
estimator_path=BUNDLE/'remove_estimator.joblib'; calibrator_path=BUNDLE/'remove_calibrator.joblib'
joblib.dump(remove_estimator,estimator_path); joblib.dump(remove_calibrator,calibrator_path)
require(hasattr(remove_estimator,'predict_proba'),'Estimator lacks predict_proba.'); require(hasattr(remove_calibrator,'predict'),'Calibrator lacks predict.')
cal_record={'class':'IsotonicRegression','out_of_bounds':'clip','fit_input':EXPECTED_RAW_REMOVE_COLUMN,'fit_target':f'{label_col} == REMOVE','fit_rows':len(df),'evidence_role':'complete Notebook 50 source-group OOF raw REMOVE scores','x_thresholds':remove_calibrator.X_thresholds_.tolist(),'y_thresholds':remove_calibrator.y_thresholds_.tolist()}
atomic_json(TRAINING/'calibrator_fitting_record.json',cal_record)
print('Estimator SHA-256:',sha256_file(estimator_path)); print('Calibrator SHA-256:',sha256_file(calibrator_path))

## 6. Freeze SAFE_REMOVE policy and region assertion contract

In [ ]:
policy={'policy_name':'M5_REMOVE_FULL_DEVELOPMENT','policy_basis':'Notebook 50 REMOVE-only decision frozen by Notebook 52','source_notebook52_policy_id':policy52['policy_id'],'selected_target':'REMOVE','selected_model_family':'local_nonlinear','safe_protect_runtime_enabled':False,'m3_fallback':True,'connectivity':8,'remove_probability_threshold':0.995,'remove_region_minimum_mean_probability':0.995,'remove_region_minimum_pixel_fraction_above_threshold':1.00,'minimum_remove_run':2,'estimator_role':'post-development all-permitted-data deployment refit','calibration_role':'complete Notebook 50 source-group OOF REMOVE scores','clean_truth_required_at_inference':False}
policy['policy_id']=object_hash(policy); atomic_json(BUNDLE/'policy.json',policy)

def assert_safe_remove_regions(frame,probability_column='p_remove'):
    require({'page','y','x',probability_column}.issubset(frame.columns),'Region assertion input incomplete.'); require(not frame.duplicated(['page','y','x']).any(),'Duplicate region assertion coordinates.')
    asserted=np.zeros(len(frame),bool); region_ids=np.full(len(frame),'',object); rows=[]
    for page,g in frame.groupby('page',sort=False):
        idx=g.index.to_numpy(); ys=g.y.to_numpy(int); xs=g.x.to_numpy(int); probs=g[probability_column].to_numpy(float)
        m=np.zeros((int(ys.max())+1,int(xs.max())+1),np.uint8); candidate=probs>=REMOVE_PROB_THRESHOLD; m[ys[candidate],xs[candidate]]=1
        n,lab,stats,_=cv2.connectedComponentsWithStats(m,CONNECTIVITY); lookup={(int(frame.at[i,'y']),int(frame.at[i,'x'])):i for i in idx}
        for cid in range(1,n):
            cy,cx=np.where(lab==cid); ids=[lookup[(int(a),int(b))] for a,b in zip(cy,cx) if (int(a),int(b)) in lookup]
            if not ids: continue
            q=frame.loc[ids,probability_column].to_numpy(float); passes=len(ids)>=MIN_REMOVE_RUN and float(q.mean())>=REMOVE_REGION_MIN_MEAN and float((q>=REMOVE_PROB_THRESHOLD).mean())>=REMOVE_REGION_MIN_PIXEL_FRACTION
            rid=f'{page}:REMOVE:r{cid:05d}'
            if passes: asserted[ids]=True; region_ids[ids]=rid
            rows.append({'page':str(page),'region_id':rid,'pixels':len(ids),'mean_probability':float(q.mean()),'minimum_probability':float(q.min()),'passes':bool(passes)})
    return asserted,region_ids,pd.DataFrame(rows)
print(json.dumps(policy,indent=2))

## 7. Deterministic replay fixture

In [ ]:
ordered=df.sort_values(['source_group','page','y','x']).reset_index(drop=True); pos=sorted(set([0,len(ordered)//4,len(ordered)//2,(3*len(ordered))//4,len(ordered)-1])); fixture=ordered.iloc[pos].copy()
fixture_X=fixture[predictors].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan); fixture_raw=remove_estimator.predict_proba(fixture_X)[:,1]; fixture_p=remove_calibrator.predict(fixture_raw)
fixture_out=fixture[['page','source_group','y','x']].copy(); fixture_out['raw_remove_score']=fixture_raw; fixture_out['p_remove']=fixture_p
fixture_out.to_csv(REPLAY/'replay_fixture.csv',index=False); fixture_X.to_csv(REPLAY/'replay_predictors.csv',index=False)
loaded_estimator=joblib.load(estimator_path); loaded_calibrator=joblib.load(calibrator_path); replay_raw=loaded_estimator.predict_proba(fixture_X)[:,1]; replay_p=loaded_calibrator.predict(replay_raw)
replay_checks={'estimator_reload_raw_scores_exact':bool(np.array_equal(fixture_raw,replay_raw)),'calibrator_reload_probabilities_exact':bool(np.array_equal(fixture_p,replay_p)),'fixture_predictor_columns_exact':list(fixture_X.columns)==list(predictors),'probabilities_finite':bool(np.isfinite(replay_p).all()),'probabilities_in_unit_interval':bool(((replay_p>=0)&(replay_p<=1)).all())}
replay={'policy_id':policy['policy_id'],'estimator_sha256':sha256_file(estimator_path),'calibrator_sha256':sha256_file(calibrator_path),'predictor_schema_sha256':sha256_file(BUNDLE/'predictor_schema.json'),'fixture_rows':len(fixture),'fixture_csv_sha256':sha256_file(REPLAY/'replay_fixture.csv'),'fixture_predictors_sha256':sha256_file(REPLAY/'replay_predictors.csv'),'checks':replay_checks,'passed':all(replay_checks.values())}
atomic_json(REPLAY/'replay_test.json',replay); require(replay['passed'],f'Replay failed: {replay}'); print(json.dumps(replay,indent=2))

## 8. Diagnostic comparison with historical OOF decisions

In [ ]:
full_raw=remove_estimator.predict_proba(X)[:,1]; full_p=remove_calibrator.predict(full_raw); audit_frame=df[['page','source_group','y','x']].copy(); audit_frame['p_remove_full_development']=full_p
full_safe,full_region_id,full_regions=assert_safe_remove_regions(audit_frame,'p_remove_full_development'); full_regions.to_csv(AUDIT/'full_development_regions_on_development_population.csv',index=False)
comparison={'rows':len(df),'full_development_safe_remove_on_development_rows':int(full_safe.sum()),'interpretation':'Diagnostic only. OOF and all-development fits have different training exposure and are not required to match.'}
if 'safe_remove' in df.columns:
    hist=df.safe_remove.astype(bool).to_numpy(); comparison.update({'historical_oof_safe_remove':int(hist.sum()),'decision_xor_rows':int(np.logical_xor(hist,full_safe).sum())})
atomic_json(AUDIT/'oof_vs_full_development_comparison.json',comparison); require(np.isfinite(full_raw).all() and np.isfinite(full_p).all(),'Non-finite full-fit outputs.'); print(json.dumps(comparison,indent=2))

## 9. Content-addressed bundle and integration record

In [ ]:
fitting={'created_utc':datetime.now(timezone.utc).isoformat(),'source_notebook50_run':str(N50),'source_notebook52_run':str(N52),'source_notebook52_policy_id':policy52['policy_id'],'source_notebook50_identity_sha256':sha256_file(N50/'experiment_identity.json'),'source_notebook50_verdict_sha256':sha256_file(N50/'notebook50_verdict.json'),'source_selected_models_sha256':sha256_file(N50/'selected_models.json'),'source_predictor_schema_sha256':sha256_file(N50/'inputs'/'predictor_schema.json'),'source_training_table_sha256':sha256_file(source_table),'selected_model_label':'local_nonlinear','constructor_record_sha256':sha256_file(TRAINING/'exact_model_constructor.json'),'training_rows':len(df),'training_pages':df.page.astype(str).nunique(),'training_source_groups':df.source_group.astype(str).nunique(),'calibration_scores':EXPECTED_RAW_REMOVE_COLUMN,'calibration_method':'IsotonicRegression(out_of_bounds=clip)','calibration_scores_are_source_group_oof':True,'full_development_estimator_is_newly_fitted':True,'thresholds_or_spatial_policy_changed':False,'safe_protect_enabled':False,'software':{'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'pandas':pd.__version__,'scikit_learn':sklearn.__version__,'joblib':joblib.__version__,'opencv':cv2.__version__}}
atomic_json(TRAINING/'fitting_manifest.json',fitting)
bundle={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.8F-full-development-remove-freeze','policy_id':policy['policy_id'],'source_notebook52_policy_id':policy52['policy_id'],'source_notebook50_run':str(N50),'source_notebook52_run':str(N52),'estimator':'remove_estimator.joblib','estimator_sha256':sha256_file(estimator_path),'calibrator':'remove_calibrator.joblib','calibrator_sha256':sha256_file(calibrator_path),'predictor_schema':'predictor_schema.json','predictor_schema_sha256':sha256_file(BUNDLE/'predictor_schema.json'),'policy':'policy.json','policy_sha256':sha256_file(BUNDLE/'policy.json'),'replay_test':'replay/replay_test.json','replay_test_sha256':sha256_file(REPLAY/'replay_test.json'),'fitting_manifest':'training_record/fitting_manifest.json','fitting_manifest_sha256':sha256_file(TRAINING/'fitting_manifest.json'),'unseen_page_scoring_available':True,'requires_exact_notebook50_observable_feature_builder':True,'promotion_authority':False}
bundle['bundle_id']=object_hash({k:bundle[k] for k in ['version','policy_id','source_notebook52_policy_id','estimator_sha256','calibrator_sha256','predictor_schema_sha256','policy_sha256','replay_test_sha256','fitting_manifest_sha256']}); atomic_json(BUNDLE/'bundle_manifest.json',bundle)
rows=[]
for p in sorted(BUNDLE.rglob('*')):
    if p.is_file() and p.name!='artifact_inventory.csv': rows.append({'relative_path':str(p.relative_to(BUNDLE)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(rows).to_csv(BUNDLE/'artifact_inventory.csv',index=False)
integration={'bundle_id':bundle['bundle_id'],'bundle_directory':str(BUNDLE),'recommended_operator_resolution':'Verify bundle_manifest hashes; load remove_estimator.joblib and remove_calibrator.joblib; build exact ordered Notebook 50 predictors; score predict_proba[:,1]; calibrate; apply policy.json page-global region rule.','notebook53_change':'Resolve this bundle explicitly instead of filename searching Notebook 50.','notebook72_capability':{'unseen_page_inference':True,'registered_replay':True}}
atomic_json(OUT/'integration_record.json',integration); print(json.dumps(bundle,indent=2))

## 10. Fail-closed verdict and final upload package

In [ ]:
final_gates={'source_controls_exact':all(checks.values()),'selected_remove_model_exact':selected_models.get('REMOVE')=='local_nonlinear','ordered_predictor_schema_preserved':read_json(BUNDLE/'predictor_schema.json').get('ordered_predictors')==predictors,'full_development_estimator_loadable':hasattr(joblib.load(estimator_path),'predict_proba'),'oof_calibrator_loadable':hasattr(joblib.load(calibrator_path),'predict'),'estimator_hash_recorded':sha256_file(estimator_path)==bundle['estimator_sha256'],'calibrator_hash_recorded':sha256_file(calibrator_path)==bundle['calibrator_sha256'],'replay_exact':read_json(REPLAY/'replay_test.json').get('passed') is True,'safe_protect_disabled':policy['safe_protect_runtime_enabled'] is False,'frozen_remove_policy_unchanged':policy['connectivity']==8 and policy['minimum_remove_run']==2 and policy['remove_probability_threshold']==0.995 and policy['remove_region_minimum_mean_probability']==0.995 and policy['remove_region_minimum_pixel_fraction_above_threshold']==1.00}
first_failed=next((k for k,v in final_gates.items() if v is not True),None); verdict='PASS_FULL_DEVELOPMENT_REMOVE_BUNDLE_READY_FOR_OPERATOR_INTEGRATION' if first_failed is None else 'FAIL_FULL_DEVELOPMENT_REMOVE_FREEZE'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':verdict,'first_failed_gate':first_failed,'gates':final_gates,'bundle_id':bundle['bundle_id'],'bundle_directory':str(BUNDLE),'meaning':'PASS establishes a content-addressed full-development SAFE_REMOVE scorer/calibrator bundle for operator integration. It is not untouched independent validation or production authorisation.'}
atomic_json(OUT/'notebook52f_verdict.json',record); pd.DataFrame([{'gate':k,'passed':v} for k,v in final_gates.items()]).to_csv(AUDIT/'notebook52f_gate_results.csv',index=False)
summary=['NOTEBOOK 52F - FULL-DEVELOPMENT SAFE_REMOVE FIT AND FREEZE','',f'Verdict: {verdict}',f'Bundle ID: {bundle["bundle_id"]}',f'Notebook 50 source: {N50}',f'Notebook 52 source: {N52}',f'Development rows: {len(df)}',f'Pages: {df.page.astype(str).nunique()}',f'Source groups: {df.source_group.astype(str).nunique()}',f'Predictors: {len(predictors)}','Model: exact Notebook 50 local_nonlinear constructor','Calibration: isotonic on complete source-group OOF raw REMOVE scores','SAFE_REMOVE: p>=0.995; region mean>=0.995; 8-connectivity; run>=2; all region pixels above threshold','SAFE_PROTECT: disabled','Unseen-page scoring: available with the exact Notebook 50 observable predictor builder','','This is a post-development deployment refit; it does not claim equality with historical OOF decisions.']
(OUT/'executive_summary.txt').write_text('
'.join(summary))
upload_files=[OUT/'executive_summary.txt',OUT/'notebook52f_verdict.json',OUT/'integration_record.json',BUNDLE/'bundle_manifest.json',BUNDLE/'policy.json',BUNDLE/'predictor_schema.json',BUNDLE/'artifact_inventory.csv',REPLAY/'replay_test.json',TRAINING/'fitting_manifest.json',TRAINING/'exact_model_constructor.json',AUDIT/'notebook52f_gate_results.csv',AUDIT/'oof_vs_full_development_comparison.json']
for p in upload_files: require(p.exists(),f'Missing upload artefact: {p}')
pd.DataFrame([{'relative_path':str(p.relative_to(OUT)),'bytes':p.stat().st_size,'sha256':sha256_file(p)} for p in upload_files]).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('
'.join(summary)+'

Binary estimator and calibrator remain in the full bundle directory and are hash-addressed by bundle_manifest.json.')
with zipfile.ZipFile(UPLOAD/'notebook52f_review_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    z.write(UPLOAD/'README.txt','README.txt'); z.write(UPLOAD/'upload_manifest.csv','upload_manifest.csv')
    for p in upload_files: z.write(p,str(p.relative_to(OUT)))
print(json.dumps(record,indent=2)); print('
'.join(summary)); print('Review ZIP:',UPLOAD/'notebook52f_review_bundle.zip')
if first_failed: raise RuntimeError(f'Notebook 52F failed closed at {first_failed}.')

## Outputs to inspect first
- `executive_summary.txt`
- `notebook52f_verdict.json`
- `integration_record.json`
- `full_development_remove/bundle_manifest.json`
- `full_development_remove/policy.json`
- `full_development_remove/predictor_schema.json`
- `full_development_remove/replay/replay_test.json`
- `full_development_remove/training_record/fitting_manifest.json`
- `audit/oof_vs_full_development_comparison.json`
- `upload/notebook52f_review_bundle.zip`